# 2HS 数值分析

以连续单孤子和二孤子为参考，比较可积半离散格式（Integrable）与固定网格中心差分（FD）。两种空间系统采用相同的 RK4 时间步长，场值在公共物理网格上重构后评价误差。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 计算使用 IEEE 754 双精度。
np.set_printoptions(precision=12)
pd.set_option("display.precision", 12)


## 1　单孤子与初始网格

考虑 $c=1$ 的 2HS 系统

$$u_{tx}-uu_{xx}-\frac12u_x^2-4u-\frac12(\rho^2-1)=0,\qquad \rho_t=(u\rho)_x.\tag{1}$$

取原文图 1、图 3 的参数 $p=5$、$q=1.25$、$\xi_0=\eta_0=0$，相应的连续单孤子解为

$$\theta=\frac{15}{4}X-\frac35t,\qquad
x=X-\frac12+\frac3{10}\tanh\frac\theta2,\qquad
u=\frac9{100}\operatorname{sech}^2\frac\theta2,\qquad
\rho=\frac{16}{25-9\tanh^2(\theta/2)}.\tag{2}$$

其中 $\rho$ 的远场值为 $1$。比较可积半离散格式（Integrable）与固定网格中心差分（FD）。取辅助节点 $X_k=-4+ka$，$a=0.005$、$k=0,\ldots,N$、$N=1600$；由式（2）给出初始物理节点与场值。FD 在相同的初始物理端点间均匀布点，物理步长记为 $\Delta x$。

In [ ]:
# 式（2）：p=5、q=1.25、c=1，初相位为零。
# 加密同一辅助区间时，同时增大 N、减小 a。
hs_cfg = dict(
    N=1600, a=0.005, Xleft=-4.0,
    dt=0.003125, T=0.5,
    xMin=-1.0, xMax=1.0, evaluationPoints=32001,
)
hs_N, hs_a, hs_Xleft = (hs_cfg[k] for k in ("N", "a", "Xleft"))
hs_dt, hs_T = (hs_cfg[k] for k in ("dt", "T"))
hs_xmin, hs_xmax, hs_neval = (hs_cfg[k] for k in ("xMin", "xMax", "evaluationPoints"))
if not np.isfinite([hs_a, hs_Xleft, hs_dt, hs_T, hs_xmin, hs_xmax]).all():
    raise ValueError("网格、时间和评价区间须取有限实数。")
if not (hs_a > 0 and hs_dt > 0 and hs_T >= 0 and hs_xmax > hs_xmin):
    raise ValueError("要求 a>0、dt>0、T≥0、xMax>xMin。")
if not isinstance(hs_N, int) or not 20 <= hs_N <= 6400:
    raise ValueError("N 应为 20—6400 的整数。")
if not isinstance(hs_neval, int) or not 101 <= hs_neval <= 64001:
    raise ValueError("评价点数应为 101—64001 的整数。")
hs_steps = round(hs_T / hs_dt)
if abs(hs_steps * hs_dt - hs_T) > 1e-12 or hs_steps > 3200:
    raise ValueError("T 须为 dt 的整数倍，且步数≤3200。")
hs_Xright = hs_Xleft + hs_N * hs_a
print(pd.DataFrame([{
    "N": hs_N, "a": hs_a, "X 左端": hs_Xleft, "X 右端": hs_Xright,
    "dt": hs_dt, "T": hs_T, "时间步数": hs_steps, "评价点数": hs_neval,
}]).to_string(index=False))


辅助坐标 $X$ 到物理坐标 $x$ 的映射严格单调。由 $J=\partial_Xx>0$ 反解物理坐标，并用链式法则得到 FD 边界所需的 $m=u_{xx}+2$。

In [ ]:
def hs_exact_X(X, t):
    """式（2）的解析场；X 可以是标量或数组。"""
    X = np.asarray(X, dtype=float)
    z = np.tanh((3.75 * X - 0.6 * t) / 2)
    S = 1 - z * z
    zX = 1.875 * S
    zXX = -2 * zX * 1.875 * z
    J, JX = 1 + 0.3 * zX, 0.3 * zXX
    u, uX = 0.09 * S, -0.18 * z * zX
    uXX = -0.18 * (zX * zX + z * zXX)
    # 链式法则：d/dx=(1/J)d/dX，其中 J=dx/dX。
    uxx = (uXX * J - uX * JX) / (J * J * J)
    return dict(x=X - 0.5 + 0.3 * z, u=u, rho=1 / J, m=uxx + 2, X=X)

def hs_exact_physical(x, t):
    """利用 dx/dX>0，在 [x+0.2,x+0.8] 中二分反解 X。"""
    x = np.asarray(x, dtype=float)
    lo, hi = x + 0.2, x + 0.8
    for _ in range(60):
        mid = (lo + hi) / 2
        lower = hs_exact_X(mid, t)["x"] < x
        lo, hi = np.where(lower, mid, lo), np.where(lower, hi, mid)
    value = hs_exact_X((lo + hi) / 2, t)
    if np.max(np.abs(value["x"] - x)) > 2e-12:
        raise ArithmeticError("物理坐标反解未达到容差。")
    return value

hs_initial = hs_exact_X(hs_Xleft + np.arange(hs_N + 1) * hs_a, 0)
hs_left, hs_right = hs_initial["x"][[0, -1]]
hs_dx = (hs_right - hs_left) / hs_N
# 固定网格与初始动网格共用物理端点。
hs_fixed_x = hs_left + np.arange(hs_N + 1) * hs_dx
if hs_xmin <= hs_left or hs_xmax >= hs_right:
    raise ValueError("评价区间必须位于初始物理端点内部。")
print(pd.DataFrame([{
    "初始物理左端": hs_left, "初始物理右端": hs_right,
    "FD 的 dx": hs_dx, "峰值 u": 0.09, "谷值 rho": 0.64,
}]).to_string(index=False))


## 2　空间离散

**Integrable。** 令 $d_k=x_{k+1}-x_k$、$b_k=u_{k+1}-u_k$，采用原文式（100）的等价形式

$$\begin{aligned}
\dot d_k&=-b_k,\qquad \dot x_0=-u_0(t),\\
\dot b_k&=2d_k(u_{k+1}+u_k)
+\frac{[d_k^2-a(a-1)]^2-b_k^2}{2d_k}-\frac{d_k}{2}.
\end{aligned}\tag{3}$$

左端 $u_0(t)$ 取解析值；各节点及胞元密度由下式递推恢复：

$$u_{k+1}=u_k+b_k,\qquad x_{k+1}=x_k+d_k,\qquad
\rho_{k+1/2}=\frac{a}{d_k}.\tag{4}$$

**FD。** 对式（1）的第一式作 $x$ 微分，令 $m=u_{xx}+2$。在固定网格上定义

$$ (D_1f)_i=\frac{f_{i+1}-f_{i-1}}{2\Delta x},\qquad
(D_2f)_i=\frac{f_{i+1}-2f_i+f_{i-1}}{\Delta x^2},\tag{5}$$

并计算

$$\begin{aligned}
\dot m_i&=u_i(D_1m)_i+2m_i(D_1u)_i+\rho_i(D_1\rho)_i,\\
\dot\rho_i&=u_i(D_1\rho)_i+\rho_i(D_1u)_i,\\
u_{i+1}-2u_i+u_{i-1}&=\Delta x^2(m_i-2).
\end{aligned}\tag{6}$$

初始取 $m_i^0=(D_2u^0)_i+2$、$\rho_i^0=\rho(x_i,0)$。两端 $u,\rho,m$ 取解析值，每个时间级由最后一式联立求出内部 $u_i$。

In [ ]:
def hs_integrable_fields(t, state):
    # 状态：[b_0,...,b_(N-1), d_0,...,d_(N-1), x_0]。
    b, d = state[:hs_N], state[hs_N:2 * hs_N]
    if not np.isfinite(state).all() or np.any(d <= 0):
        raise ArithmeticError("Integrable 出现非有限状态或非正胞元格距。")
    u0 = hs_exact_X(hs_Xleft, t)["u"]
    u = np.r_[u0, u0 + np.cumsum(b)]
    x = np.r_[state[-1], state[-1] + np.cumsum(d)]
    return dict(x=x, u=u, rhoX=(x[:-1] + x[1:]) / 2, rho=hs_a / d)

def hs_integrable_rhs(t, state):
    # 式（3）：b、d 和 x_0 的时间导数。
    u = hs_integrable_fields(t, state)["u"]
    b, d = state[:hs_N], state[hs_N:2 * hs_N]
    C = d * d - hs_a * (hs_a - 1)
    db = 2 * d * (u[1:] + u[:-1]) + (C * C - b * b) / (2 * d) - d / 2
    return np.r_[db, -b, -u[0]]

def hs_recover_u(m, ub_left, ub_right):
    # Thomas 消元：u_(i-1)-2u_i+u_(i+1)=dx^2*(m_i-2)。
    count = hs_N - 1
    c, d = np.zeros(count), np.zeros(count)
    for j in range(count):
        diagonal = -2 - (c[j - 1] if j else 0)
        rhs = hs_dx * hs_dx * (m[j + 1] - 2)
        if j == 0:
            rhs -= ub_left
        if j == count - 1:
            rhs -= ub_right
        c[j] = 1 / diagonal if j < count - 1 else 0
        d[j] = (rhs - (d[j - 1] if j else 0)) / diagonal
    u = np.empty(hs_N + 1)
    u[0], u[-1] = ub_left, ub_right
    for j in range(count - 1, -1, -1):
        u[j + 1] = d[j] - (c[j] * u[j + 2] if j < count - 1 else 0)
    return u

def hs_fd_fields(t, state):
    # FD 状态为内部 m 与 rho，端点取解析边界。
    boundary = hs_exact_physical(np.array([hs_left, hs_right]), t)
    m = np.r_[boundary["m"][0], state[:hs_N - 1], boundary["m"][1]]
    rho = np.r_[boundary["rho"][0], state[hs_N - 1:], boundary["rho"][1]]
    if not np.isfinite(m).all() or not np.isfinite(rho).all() or np.any(rho <= 0):
        raise ArithmeticError("FD 出现非有限状态或非正密度。")
    u = hs_recover_u(m, *boundary["u"])
    return dict(x=hs_fixed_x, u=u, rhoX=hs_fixed_x, rho=rho, m=m)

def hs_fd_rhs(t, state):
    # 式（6）：固定网格上的一阶中心差分。
    fields = hs_fd_fields(t, state)
    m, rho, u = (fields[k] for k in ("m", "rho", "u"))
    ux = (u[2:] - u[:-2]) / (2 * hs_dx)
    mx = (m[2:] - m[:-2]) / (2 * hs_dx)
    rx = (rho[2:] - rho[:-2]) / (2 * hs_dx)
    return np.r_[u[1:-1] * mx + 2 * m[1:-1] * ux + rho[1:-1] * rx,
                 u[1:-1] * rx + rho[1:-1] * ux]

def hs_initial_states():
    # 初始 m 使用与演化一致的二阶差分模板。
    fields = hs_exact_physical(hs_fixed_x, 0)
    u0 = fields["u"]
    integrable = np.r_[np.diff(hs_initial["u"]), np.diff(hs_initial["x"]), hs_left]
    m0 = (u0[2:] - 2 * u0[1:-1] + u0[:-2]) / (hs_dx * hs_dx) + 2
    fd = np.r_[m0, fields["rho"][1:-1]]
    return integrable, fd

def hs_interpolate(xs, values, targets):
    # 物理节点上的分段线性重构。
    if targets[0] < xs[0] or targets[-1] > xs[-1]:
        raise ValueError("数值网格未覆盖整个评价区间。")
    # side='left' 与原报告在恰好落于节点时采用相同的左区间。
    j = np.clip(np.searchsorted(xs, targets, side="left") - 1, 0, len(xs) - 2)
    weight = (targets - xs[j]) / (xs[j + 1] - xs[j])
    return values[j] + weight * (values[j + 1] - values[j])


## 3　时间推进

两种系统均用 RK4 更新。记上述演化方程为 $\dot z=\mathcal F(t,z)$，则

$$\begin{aligned}
K_1&=\mathcal F(t_n,z^n),&
K_2&=\mathcal F\!\left(t_n+\frac{\Delta t}{2},z^n+\frac{\Delta t}{2}K_1\right),\\
K_3&=\mathcal F\!\left(t_n+\frac{\Delta t}{2},z^n+\frac{\Delta t}{2}K_2\right),&
K_4&=\mathcal F(t_n+\Delta t,z^n+\Delta tK_3),\\
z^{n+1}&=z^n+\frac{\Delta t}{6}(K_1+2K_2+2K_3+K_4).
\end{aligned}\tag{7}$$

每一级均按式（4）或（28）恢复物理场。取 $\Delta t=0.003125$，从 $t=0$ 计算至 $t=0.5$。

In [ ]:
def hs_rk4_step(rhs, t, state, delta):
    # 式（7）：每一级重新恢复场值和解析边界。
    k1 = rhs(t, state)
    k2 = rhs(t + delta / 2, state + delta / 2 * k1)
    k3 = rhs(t + delta / 2, state + delta / 2 * k2)
    k4 = rhs(t + delta, state + delta * k3)
    return state + delta * (k1 + 2 * k2 + 2 * k3 + k4) / 6


两种方案从各自的解析初值推进到同一时刻 $T$。Integrable 的物理网格随 $\dot x=-u$ 运动，FD 的节点位置保持固定。

In [ ]:
hs_integrable, hs_fd = hs_initial_states()
# 时间层 t_n=n*dt；再次执行本单元会从当前配置的初值开始。
for hs_step in range(hs_steps):
    hs_t = hs_step * hs_dt
    hs_integrable = hs_rk4_step(hs_integrable_rhs, hs_t, hs_integrable, hs_dt)
    hs_fd = hs_rk4_step(hs_fd_rhs, hs_t, hs_fd, hs_dt)
hs_live_fields = {
    "Integrable": hs_integrable_fields(hs_T, hs_integrable),
    "FD": hs_fd_fields(hs_T, hs_fd),
}
print(pd.DataFrame([
    {"方案": name, "已推进步数": hs_steps, "当前时间": hs_T, "物理节点数": len(f["x"])}
    for name, f in hs_live_fields.items()
]).to_string(index=False))


## 4　单孤子误差与波形

在 $x\in[-1,1]$ 的 32001 个等距点组成的集合 $\mathcal G$ 上，经线性插值后计算相对连续解的最大绝对误差：

$$E_u(t)=\max_{x\in\mathcal G}|u_h(x,t)-u(x,t)|,\qquad
E_\rho(t)=\max_{x\in\mathcal G}|\rho_h(x,t)-\rho(x,t)|.\tag{8}$$



密度分别位于 Integrable 的胞元中点与 FD 的节点。误差图每段宽 $0.02$，取段内评价点的最大绝对误差；对数图的显示下限不改变误差范数。

In [ ]:
# 式（8）：公共物理网格上的最大绝对误差。
hs_xx = hs_xmin + (hs_xmax - hs_xmin) * np.arange(hs_neval) / (hs_neval - 1)
hs_reference = hs_exact_physical(hs_xx, hs_T)
hs_results, hs_result_rows = {}, []
for hs_name, hs_fields in hs_live_fields.items():
    hs_u = hs_interpolate(hs_fields["x"], hs_fields["u"], hs_xx)
    hs_rho = hs_interpolate(hs_fields["rhoX"], hs_fields["rho"], hs_xx)
    hs_eu = np.abs(hs_u - hs_reference["u"])
    hs_er = np.abs(hs_rho - hs_reference["rho"])
    hs_r = dict(Eu=hs_eu.max(), Erho=hs_er.max(), eu=hs_eu, er=hs_er,
                u=hs_u, rho=hs_rho, minDx=np.diff(hs_fields["x"]).min(),
                minRho=hs_fields["rho"].min())
    hs_results[hs_name] = hs_r
    hs_result_rows.append({"方案": hs_name, "E_u(T)": hs_r["Eu"],
                           "E_rho(T)": hs_r["Erho"], "终点最小格距": hs_r["minDx"],
                           "终点最小密度": hs_r["minRho"]})
print(pd.DataFrame(hs_result_rows).to_string(index=False))

# 每个绘图区间保留其评价点上的最大误差。
hs_bins = 100
hs_centers = hs_xmin + (np.arange(hs_bins) + 0.5) * (hs_xmax - hs_xmin) / hs_bins
hs_bin_index = np.minimum(hs_bins - 1, np.floor(
    (hs_xx - hs_xmin) / (hs_xmax - hs_xmin) * hs_bins).astype(int))
for hs_key, hs_label in (("eu", "u"), ("er", r"\rho")):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for hs_name, hs_style in (("Integrable", "-"), ("FD", "--")):
        hs_envelope = np.zeros(hs_bins)
        np.maximum.at(hs_envelope, hs_bin_index, hs_results[hs_name][hs_key])
        # 对数图的显示下限为 1e-16；误差范数不截断。
        ax.plot(hs_centers, np.log10(np.maximum(hs_envelope, 1e-16)),
                hs_style, label=hs_name)
    ax.set(xlabel=r"Physical coordinate $x$",
           ylabel=rf"$\log_{{10}}\max|{hs_label}_h-{hs_label}|$")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

hs_single_summary = {name: dict(Eu=float(r['Eu']), Erho=float(r['Erho']))
                     for name, r in hs_results.items()}


在同一物理区间比较数值场与连续解，误差曲线对应上表的最大绝对误差。

In [ ]:
def hs_plot_waveforms(fields, reference, title):
    fig, axes = plt.subplots(2, 2, figsize=(9, 5.5), sharex=True)
    for column, (name, values) in enumerate(fields.items()):
        for row, key in enumerate(("u", "rho")):
            ax = axes[row, column]
            coordinates = values["x"] if key == "u" else values["rhoX"]
            ax.plot(hs_xx, reference[key], "-", color="firebrick", label="Exact")
            mask = (coordinates >= hs_xmin) & (coordinates <= hs_xmax)
            # 稀疏显示数值节点，完整节点仍参与误差评价。
            ids = np.flatnonzero(mask)[::max(1, int(mask.sum()) // 100)]
            ax.plot(coordinates[ids], values[key][ids], ".", markersize=3,
                    label=name, color="royalblue")
            ax.set_title(name if row == 0 else "")
            ax.set_ylabel(r"$u$" if key == "u" else r"$\rho$")
            if row == 1:
                ax.set_xlabel(r"Physical coordinate $x$")
            ax.grid(alpha=.2)
            ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

hs_plot_waveforms(hs_live_fields, hs_reference, f"2HS one soliton, t={hs_T:g}")


## 5　二孤子

取 $p_1=1.1$、$p_2=1.25$、$q_1=11$、$q_2=5$、$c=1$，相位常数均为 $\log6.5$。
连续二孤子的 $\tau$ 函数为

$$\begin{aligned}
f&=1+e^{\theta_1}+e^{\theta_2}+A_{12}e^{\theta_1+\theta_2},
\qquad A_{12}=\frac{12}{507},\\
\theta_i&=(p_i-q_i)X+\left(\frac1{p_i}-\frac1{q_i}\right)t+\log6.5.
\end{aligned}\tag{9}$$

相应的物理坐标与场为

$$u=\partial_t^2\log f,\qquad
x=X-\partial_t\log f-\frac1{q_1}-\frac1{q_2},\qquad
\rho=\frac1{1-\partial_X\partial_t\log f}.\tag{10}$$

归一化后的四项指数构成权重。$u$ 是时间指数系数的方差，
$J=\partial x/\partial X$ 由时间、空间指数系数的协方差给出；
进一步的混合导数确定 $m=u_{xx}+2$，供 FD 边界使用。


In [ ]:
# 四项 τ 函数的指数系数；q_i=p_i/(p_i-1) 对应 c=1。
hs_two_p = np.array([1.1, 1.25])
hs_two_q = np.array([11.0, 5.0])
hs_two_phase = np.log(6.5)
hs_two_A = 12 / 507
if not np.allclose(hs_two_q, hs_two_p / (hs_two_p - 1)):
    raise ValueError("p、q 必须满足 c=1 的孤子参数约束。")
hs_two_beta = np.r_[0, hs_two_p-hs_two_q, np.sum(hs_two_p-hs_two_q)][:, None]
hs_two_omega = np.r_[0, 1/hs_two_p-1/hs_two_q,
                     np.sum(1/hs_two_p-1/hs_two_q)][:, None]
hs_two_logcoef = np.array([0, hs_two_phase, hs_two_phase,
                          2*hs_two_phase + np.log(hs_two_A)])[:, None]
hs_two_shift = float(np.sum(1/hs_two_q))

def hs_two_exact_X(X, t):
    """u、J=dx/dX 和 m=u_xx+2 均由 τ 的解析导数计算。"""
    X = np.asarray(X, dtype=float)
    flat = X.reshape(-1)
    logs = hs_two_beta*flat + hs_two_omega*t + hs_two_logcoef
    # 减去列最大值避免指数溢出，不改变归一化权重。
    weights = np.exp(logs - np.max(logs, axis=0))
    weights /= np.sum(weights, axis=0)
    mean_o = np.sum(weights*hs_two_omega, axis=0)
    mean_b = np.sum(weights*hs_two_beta, axis=0)
    do, db = hs_two_omega-mean_o, hs_two_beta-mean_b
    u = np.sum(weights*do**2, axis=0)
    cov = np.sum(weights*do*db, axis=0)
    J = 1-cov
    JX = -np.sum(weights*do*db**2, axis=0)
    uX = np.sum(weights*do**2*db, axis=0)
    uXX = (np.sum(weights*do**2*db**2, axis=0)
           - u*np.sum(weights*db**2, axis=0) - 2*cov**2)
    if not np.isfinite(J).all() or np.any(J <= 0):
        raise ArithmeticError("二孤子物理坐标要求 J=dx/dX>0。")
    uxx = (uXX*J-uX*JX)/J**3
    return {k: value.reshape(X.shape) for k, value in dict(
        x=flat-mean_o-hs_two_shift, u=u, rho=1/J, m=uxx+2,
        X=flat, J=J, uX=uX, uXX=uXX, JX=JX).items()}

def hs_two_exact_physical(x, t):
    """由 τ 权重的凸组合界定 X 的区间，再二分反解。"""
    x = np.asarray(x, dtype=float)
    lo = x + hs_two_shift + float(np.min(hs_two_omega))
    hi = x + hs_two_shift + float(np.max(hs_two_omega))
    for _ in range(60):
        mid = (lo+hi)/2
        lower = hs_two_exact_X(mid, t)["x"] < x
        lo, hi = np.where(lower, mid, lo), np.where(lower, hi, mid)
    value = hs_two_exact_X((lo+hi)/2, t)
    if np.max(np.abs(value["x"]-x)) > 2e-12:
        raise ArithmeticError("二孤子物理坐标反解未达到容差。")
    return value


辅助节点仍取 $X_k=-4+ka$。网格、时间步长、终点与误差评价区间沿用单孤子的配置，
初值改为二孤子的 $u(X_k,0)$、$x(X_k,0)$；FD 在相同物理端点间均匀布点。


In [ ]:
# 仅替换解析初值与边界；空间格式、时间格式和评价区间保持相同。
hs_one_results = hs_results
hs_exact_X, hs_exact_physical = hs_two_exact_X, hs_two_exact_physical
hs_initial = hs_exact_X(hs_Xleft + np.arange(hs_N+1)*hs_a, 0)
hs_left, hs_right = hs_initial["x"][[0, -1]]
hs_dx = (hs_right-hs_left)/hs_N
hs_fixed_x = hs_left + np.arange(hs_N+1)*hs_dx
if hs_xmin <= hs_left or hs_xmax >= hs_right:
    raise ValueError("评价区间必须位于二孤子初始物理端点内部。")
if np.any(np.diff(hs_initial["x"]) <= 0):
    raise ArithmeticError("二孤子初始物理网格必须严格递增。")
print(pd.DataFrame([{
    "N": hs_N, "a": hs_a, "dt": hs_dt, "T": hs_T,
    "物理左端": hs_left, "物理右端": hs_right, "FD dx": hs_dx,
    "初始最小 J": float(hs_initial["J"].min()),
}]).to_string(index=False))


二孤子采用相同的空间方程。Integrable 的初始增量为
$b_k^0=u_{k+1}^0-u_k^0$、$d_k^0=x_{k+1}^0-x_k^0$，并演化

$$\begin{aligned}
\dot d_k&=-b_k,\qquad \dot x_0=-u_0(t),\\
\dot b_k&=2d_k(u_{k+1}+u_k)
+\frac{[d_k^2-a(a-1)]^2-b_k^2}{2d_k}-\frac{d_k}{2},\\
u_{k+1}&=u_k+b_k,\qquad x_{k+1}=x_k+d_k,\qquad \rho_{k+1/2}=\frac a{d_k}.
\end{aligned}\tag{11}$$

FD 初始取 $m_i^0=(D_2u^0)_i+2$、$\rho_i^0=\rho(x_i,0)$，并演化

$$\begin{aligned}
\dot m_i&=u_i(D_1m)_i+2m_i(D_1u)_i+\rho_i(D_1\rho)_i,\\
\dot\rho_i&=u_i(D_1\rho)_i+\rho_i(D_1u)_i,\\
u_{i+1}-2u_i+u_{i-1}&=\Delta x^2(m_i-2).
\end{aligned}\tag{12}$$

每个 RK4 阶段采用二孤子的解析边界，时间更新仍取式（7）。


In [ ]:
hs_integrable, hs_fd = hs_initial_states()
hs_two_min_edge, hs_two_min_density = np.inf, np.inf
for hs_step in range(hs_steps):
    hs_t = hs_step*hs_dt
    hs_integrable = hs_rk4_step(hs_integrable_rhs, hs_t, hs_integrable, hs_dt)
    hs_fd = hs_rk4_step(hs_fd_rhs, hs_t, hs_fd, hs_dt)
    # 动网格正格距、两种方案正密度须在整个演化区间保持。
    hs_two_min_edge = min(hs_two_min_edge, float(hs_integrable[hs_N:2*hs_N].min()))
    hs_two_min_density = min(hs_two_min_density,
        float(hs_a/hs_integrable[hs_N:2*hs_N].max()),
        float(hs_fd[hs_N-1:].min()))
hs_live_fields = {
    "Integrable": hs_integrable_fields(hs_T, hs_integrable),
    "FD": hs_fd_fields(hs_T, hs_fd),
}
print(pd.DataFrame([
    {"方案": name, "时间步数": hs_steps, "终点时间": hs_T,
     "最小格距": float(np.diff(f["x"]).min()),
     "最小密度": float(f["rho"].min())}
    for name, f in hs_live_fields.items()
]).to_string(index=False))


在同一物理评价点集上按式（8）计算 $u$ 与 $\rho$ 的最大绝对误差。


In [ ]:
hs_xx = np.linspace(hs_xmin, hs_xmax, hs_neval)
hs_reference = hs_exact_physical(hs_xx, hs_T)
hs_results, hs_result_rows = {}, []
for hs_name, hs_fields in hs_live_fields.items():
    hs_u = hs_interpolate(hs_fields["x"], hs_fields["u"], hs_xx)
    hs_rho = hs_interpolate(hs_fields["rhoX"], hs_fields["rho"], hs_xx)
    hs_eu = np.abs(hs_u-hs_reference["u"])
    hs_er = np.abs(hs_rho-hs_reference["rho"])
    hs_results[hs_name] = dict(Eu=float(hs_eu.max()), Erho=float(hs_er.max()),
        eu=hs_eu, er=hs_er, u=hs_u, rho=hs_rho,
        minDx=float(np.diff(hs_fields["x"]).min()), minRho=float(hs_fields["rho"].min()))
    hs_result_rows.append({"方案": hs_name, "E_u(T)": hs_eu.max(),
                          "E_rho(T)": hs_er.max()})
hs_two_results = hs_results
hs_two_summary = {name: dict(Eu=r["Eu"], Erho=r["Erho"])
                  for name, r in hs_results.items()}
print(pd.DataFrame(hs_result_rows).to_string(index=False))


终点波形与连续二孤子作直接比较。误差曲线将评价区间等分为 100 段，
每段保留最大绝对误差；对数图的显示下限为 $10^{-16}$，误差范数本身不截断。


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for col, hs_name in enumerate(("Integrable", "FD")):
    f = hs_live_fields[hs_name]
    for row, (field, coordinates) in enumerate((("u", "x"), ("rho", "rhoX"))):
        ax = axes[row, col]
        ax.plot(hs_xx, hs_reference[field], "r-", lw=1.5, label="Exact")
        mask = (f[coordinates]>=hs_xmin) & (f[coordinates]<=hs_xmax)
        ax.plot(f[coordinates][mask], f[field][mask], "b.", ms=2, label=hs_name)
        ax.set(ylabel="$u$" if field=="u" else r"$\rho$", title=hs_name)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel(r"Physical coordinate $x$")
fig.tight_layout()
plt.show()

# 将评价区间等分为 100 段，每段取误差最大值。
hs_bins = 100
hs_centers = hs_xmin+(np.arange(hs_bins)+0.5)*(hs_xmax-hs_xmin)/hs_bins
hs_bin_index = np.minimum(hs_bins-1, np.floor(
    (hs_xx-hs_xmin)/(hs_xmax-hs_xmin)*hs_bins).astype(int))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True)
for ax, key, label in zip(axes, ("eu", "er"), ("u", r"\rho")):
    for name, style in (("Integrable", "-"), ("FD", "--")):
        envelope = np.zeros(hs_bins)
        np.maximum.at(envelope, hs_bin_index, hs_results[name][key])
        ax.plot(hs_centers, np.log10(np.maximum(envelope, 1e-16)), style, label=name)
    ax.set(xlabel=r"Physical coordinate $x$",
           ylabel=rf"$\log_{{10}}\max|{label}_h-{label}|$")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
print(pd.DataFrame([{"场": field, "Integrable / FD 误差比":
    hs_results["Integrable"][key]/hs_results["FD"][key]}
    for field, key in (("u", "Eu"), ("rho", "Erho"))]).to_string(index=False))

hs_summary_rows = []
for case, results in (("单孤子", hs_single_summary), ("二孤子", hs_two_summary)):
    for name, result in results.items():
        hs_summary_rows.append({"算例": case, "方案": name,
                                "E_u(T)": result["Eu"], "E_rho(T)": result["Erho"]})
print(pd.DataFrame(hs_summary_rows).to_string(index=False))


这里比较的是连续解析初值下的终点场误差，包含空间离散、RK4 时间推进、边界与线性重构的共同影响。
Integrable 采用原半离散方程的有限格距系数，FD 采用固定网格中心差分；
这一参数组的误差排序反映两种具体实现的表现。
